# Hafta 2 — Metin İşleme ve Tokenizasyon

Bir dize nasıl tamsayı dizisine dönüşür? Boşluğa göre bölmeyle başlıyor, ardından her modern alt sözcük algoritmasını adım adım yeniden kuruyoruz: Byte-Pair Encoding (BPE), WordPiece ve SentencePiece'te kullanılan Unigram dil modeli.

## Öğrenme Hedefleri

- Karakter, sözcük ve alt sözcük düzeyinde tokenizasyon arasındaki ödünleşimleri açıklamak.
- BPE'yi sıfırdan uygulamak ve gerçek bir derlem üzerinde eğitmek.
- WordPiece'i (olabilirlik tabanlı birleştirme ölçütü) ve EM ile Unigram dil modelini uygulamak.
- Sıfırdan yazılmış bir tokenizer'ı hız ve sıkıştırma oranı açısından `tokenizers` ile kıyaslamak.

## Zorunlu Okumalar

- Sennrich, R., Haddow, B. ve Birch, A. (2016). *Neural Machine Translation of Rare Words with Subword Units*.
- Kudo, T. (2018). *Subword Regularization*.
- Schuster, M. ve Nakajima, K. (2012). *Japanese and Korean Voice Search*.

In [ ]:
import sys, re, unicodedata
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

np.random.seed(0)

## 1. Normalizasyon ve ön tokenizasyon

Her tokenizer, göründüğünden çok daha önemli olan iki adımla başlar:

- **Unicode normalizasyonu.** *é* karakteri tek bir kod noktası (NFC) olarak ya da *e* + birleşen aksan (NFD) olarak kodlanabilir. Standart seçim NFC'dir.
- **Ön tokenizasyon.** BPE'nin daha sonra incelteceği kaba bir bölme. En yaygın şema: boşluklardan böl ve noktalama işaretlerini ayır.

> **Türkçe için not:** Python'un `str.lower()` yöntemi dile duyarlı değildir: `"I".lower()` Türkçedeki gibi `"ı"` değil `"i"` verir; `"İ".lower()` ise `"i"` + birleşen nokta (`U+0307`) üretir. Türkçe metinlerde önce `I → ı` ve `İ → i` dönüşümlerini elle yapmak gerekir.

In [ ]:
def normalize(text, lower=True, form='NFC'):
    text = unicodedata.normalize(form, text)
    return text.lower() if lower else text

def pre_tokenize(text):
    # Boşlukla ayrılmış parçaları böler, ardından baştaki/sondaki noktalamayı ayırır.
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)

sample = "Don't tokenize naïvely — it's harder than it looks."
print(normalize(sample))
print(pre_tokenize(normalize(sample)))

## 2. Sıfırdan Byte-Pair Encoding

BPE aslında bir sıkıştırma algoritmasıydı (Gage, 1994); NLP'ye Sennrich vd. (2016) tarafından uyarlandı.

**Eğitim algoritması.**

1. Sözlüğü derlemdeki tüm karakterlerle başlat.
2. Her sözcüğü, özel bir sözcük sonu işaretiyle `</w>` biten bir karakter dizisi olarak temsil et.
3. Derlem genelinde tüm komşu sembol çiftlerini say.
4. En sık çifti yeni bir sembolde birleştir; sözlüğe ekle.
5. Sözlük istenen $V$ boyutuna ulaşana kadar tekrarla.

**Yeni bir sözcüğü kodlama.** Öğrenilen birleştirmeleri öğrenildikleri sırayla uygula.

In [ ]:
class BPETokenizer:
    def __init__(self):
        self.merges = []                  # (a, b) çift birleştirmelerinin sıralı listesi
        self.vocab = set()

    def _word_to_symbols(self, word):
        return list(word) + ['</w>']

    def _get_pairs(self, splits):
        # splits: dict[sözcük_demeti] -> sayı
        pairs = Counter()
        for symbols, count in splits.items():
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i+1])] += count
        return pairs

    def _merge_pair(self, splits, pair):
        new_splits = {}
        a, b = pair
        for symbols, count in splits.items():
            new = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == a and symbols[i+1] == b:
                    new.append(a + b)
                    i += 2
                else:
                    new.append(symbols[i])
                    i += 1
            new_splits[tuple(new)] = count
        return new_splits

    def train(self, corpus, num_merges=200, verbose=False):
        word_counts = Counter(pre_tokenize(normalize(corpus)))
        splits = {tuple(self._word_to_symbols(w)): c for w, c in word_counts.items()}
        self.vocab = set(s for symbols in splits for s in symbols)

        for step in range(num_merges):
            pairs = self._get_pairs(splits)
            if not pairs:
                break
            best = pairs.most_common(1)[0][0]
            self.merges.append(best)
            self.vocab.add(best[0] + best[1])
            splits = self._merge_pair(splits, best)
            if verbose and step < 5:
                print(f"adım {step:3d}: birleştir {best} -> {best[0]+best[1]!r}")
        return self

    def encode_word(self, word):
        symbols = self._word_to_symbols(word)
        for a, b in self.merges:
            i = 0
            while i < len(symbols) - 1:
                if symbols[i] == a and symbols[i+1] == b:
                    symbols = symbols[:i] + [a + b] + symbols[i+2:]
                else:
                    i += 1
        return symbols

    def encode(self, text):
        out = []
        for w in pre_tokenize(normalize(text)):
            out.extend(self.encode_word(w))
        return out

# Hangi birleştirmelerin oluştuğunu görmek için çok küçük bir derlemde eğit.
toy_corpus = ("low low low low low "
              "lower lower newer newer newer newer newer newer "
              "wider wider wider widest widest")
bpe = BPETokenizer().train(toy_corpus, num_merges=10, verbose=True)
print(f"\nSözlük boyutu: {len(bpe.vocab)}")
print(f"'lowest' kodlaması: {bpe.encode('lowest')}")
print(f"'newest' kodlaması: {bpe.encode('newest')}")

In [ ]:
# Daha gerçekçi bir derlemde eğit ve sıkıştırmaya bak.
big_corpus = ("the quick brown fox jumps over the lazy dog. " * 200 +
              "tokenization is the first step in any natural language pipeline. " * 200 +
              "subword units help with out of vocabulary words. " * 200 +
              "agglutinative morphology benefits enormously from subword segmentation. " * 200)

for V in [50, 200, 500, 1000]:
    bpe = BPETokenizer().train(big_corpus, num_merges=V)
    tokens = bpe.encode("agglutinative tokenizers are extraordinarily useful.")
    print(f"birleştirme={V:4d} | sözlük={len(bpe.vocab):4d} | "
          f"{len(tokens):2d} tokena kodlandı: {tokens}")

## 3. WordPiece — olabilirlik tabanlı birleştirme

WordPiece (Schuster ve Nakajima, 2012; BERT tarafından kullanılır), BPE'nin sıklık ölçütünü bir olabilirlik ölçütüyle değiştirir. $(a, b)$ çiftinin aday birleştirmesini tanımlayalım ve $\text{count}(\cdot)$ derlemdeki sayımları göstersin. Skor şudur:

$$\text{score}(a, b) = \frac{\text{count}(ab)}{\text{count}(a) \cdot \text{count}(b)}.$$

Sezgi: ortak sıklığı, bağımsızlığın öngöreceğinden çok daha yüksek olan çiftleri birleştir — bu, logaritması alınmamış noktasal karşılıklı bilginin (PMI) ta kendisidir. Bu skoru en büyükleyen birleştirme uygulanır.

In [ ]:
class WordPieceTokenizer:
    def __init__(self):
        self.merges = []
        self.vocab = set()

    def _get_pair_and_token_counts(self, splits):
        pair_counts = Counter()
        token_counts = Counter()
        for symbols, count in splits.items():
            for s in symbols:
                token_counts[s] += count
            for i in range(len(symbols) - 1):
                pair_counts[(symbols[i], symbols[i+1])] += count
        return pair_counts, token_counts

    def _merge_pair(self, splits, pair):
        a, b = pair
        new_splits = {}
        for symbols, count in splits.items():
            new = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == a and symbols[i+1] == b:
                    new.append(a + b); i += 2
                else:
                    new.append(symbols[i]); i += 1
            new_splits[tuple(new)] = count
        return new_splits

    def train(self, corpus, num_merges=200):
        word_counts = Counter(pre_tokenize(normalize(corpus)))
        # WordPiece kuralı: sözcük içi devam parçalarının başına '##' eklenir.
        splits = {}
        for w, c in word_counts.items():
            symbols = [w[0]] + ['##' + ch for ch in w[1:]]
            splits[tuple(symbols)] = c
        self.vocab = set(s for symbols in splits for s in symbols)

        for _ in range(num_merges):
            pair_counts, tok_counts = self._get_pair_and_token_counts(splits)
            if not pair_counts:
                break
            # score = count(ab) / (count(a) * count(b))
            scores = {p: c / (tok_counts[p[0]] * tok_counts[p[1]])
                      for p, c in pair_counts.items()}
            best = max(scores, key=scores.get)
            merged = best[0] + (best[1][2:] if best[1].startswith('##') else best[1])
            self.merges.append((best, merged))
            self.vocab.add(merged)
            splits = self._merge_pair(splits, best)
        return self

    def encode_word(self, word):
        """Öğrenilen birleştirmeleri öğrenildikleri sırayla uygular (BPE'deki gibi)."""
        symbols = [word[0]] + ['##' + ch for ch in word[1:]]
        for (a, b), merged in self.merges:
            i = 0
            while i < len(symbols) - 1:
                if symbols[i] == a and symbols[i + 1] == b:
                    symbols = symbols[:i] + [merged] + symbols[i + 2:]
                else:
                    i += 1
        return symbols

wp = WordPieceTokenizer().train(big_corpus, num_merges=200)
print(f"WordPiece sözlük boyutu: {len(wp.vocab)}")
print("İlk 10 birleştirme:")
for pair, merged in wp.merges[:10]:
    print(f"  {pair} -> {merged!r}")

## 4. Unigram dil modeli (SentencePiece)

Kudo (2018), tokenizasyonu olasılıksal bir model olarak ele alır: $V$ sözlüğündeki her tokenın bir $p(t)$ olasılığı vardır ve bir $w$ sözcüğünün en iyi bölütlemesi şudur:

$$\arg\max_{s \in S(w)} \prod_{t \in s} p(t),$$

burada $S(w)$ geçerli bölütlemeler kümesidir. Eğitim:

1. Büyük bir başlangıç sözlüğüyle başla (ör. $L$ uzunluğuna kadar tüm alt dizeler).
2. $p(t)$'yi EM ile tahmin et: her sözcük için beklenen sayımları ileri algoritmasıyla (basitleştirilmiş bir varyant için yalnızca Viterbi ile) hesapla.
3. $|V|$'yi küçültmek için en düşük faydalı tokenları buda.
4. $|V|$ hedefe ulaşana kadar tekrarla.

Aşağıdaki uygulama **yalnızca Viterbi** kullanan basitleştirmedir; yine de temel fikirleri gösterir.

In [ ]:
class UnigramTokenizer:
    def __init__(self, vocab_size=100):
        self.vocab_size = vocab_size
        self.log_probs = {}

    def _seed_vocab(self, corpus, max_len=6):
        # max_len uzunluğuna kadar tüm alt dizeler, sıklıklarıyla ağırlıklandırılmış.
        counts = Counter()
        for w in pre_tokenize(normalize(corpus)):
            for i in range(len(w)):
                for j in range(i+1, min(i+max_len, len(w))+1):
                    counts[w[i:j]] += 1
        return counts

    def _viterbi_segment(self, word):
        # Geçerli log_probs altında en iyi bölütleme. (tokenlar, skor) döndürür.
        n = len(word)
        best = [(-1e18, None)] * (n + 1)
        best[0] = (0.0, None)
        for i in range(1, n + 1):
            for j in range(max(0, i - 16), i):
                sub = word[j:i]
                if sub in self.log_probs:
                    score = best[j][0] + self.log_probs[sub]
                    if score > best[i][0]:
                        best[i] = (score, (j, sub))
        # Geri izle
        if best[n][1] is None:
            return list(word), -1e18  # karakterlere geri dön
        tokens, i = [], n
        while i > 0:
            j, sub = best[i][1]
            tokens.append(sub); i = j
        return tokens[::-1], best[n][0]

    def train(self, corpus, num_iters=5, shrink_factor=0.75):
        seed = self._seed_vocab(corpus)
        total = sum(seed.values())
        self.log_probs = {t: np.log(c / total) for t, c in seed.items()}
        word_counts = Counter(pre_tokenize(normalize(corpus)))

        for it in range(num_iters):
            # E-adımı: beklenen sayımları Viterbi bölütlemesiyle topla.
            new_counts = Counter()
            for w, c in word_counts.items():
                tokens, _ = self._viterbi_segment(w)
                for t in tokens:
                    new_counts[t] += c
            total = sum(new_counts.values()) or 1
            self.log_probs = {t: np.log(c / total) for t, c in new_counts.items() if c > 0}

            # Küçült: en düşük olasılıklı tokenları at (ama tek karakterlerin hepsini tut).
            target = max(self.vocab_size, int(len(self.log_probs) * shrink_factor))
            if len(self.log_probs) > self.vocab_size:
                kept = sorted(self.log_probs.items(), key=lambda kv: kv[1], reverse=True)
                must_keep = {t for t in self.log_probs if len(t) == 1}
                pruned = {t for t, _ in kept[:target]} | must_keep
                self.log_probs = {t: lp for t, lp in self.log_probs.items() if t in pruned}
        return self

    def encode_word(self, w):
        return self._viterbi_segment(w)[0]

uni = UnigramTokenizer(vocab_size=120).train(big_corpus, num_iters=4)
print(f"Son sözlük boyutu: {len(uni.log_probs)}")
print(f"'tokenization' kodlaması: {uni.encode_word('tokenization')}")
print(f"'agglutinative' kodlaması: {uni.encode_word('agglutinative')}")

## 5. Üç yöntemi Türkçe üzerinde karşılaştırmak

Türkçe sondan eklemeli, biçimbilimsel olarak zengin bir dildir. İngilizcede birden çok sözcükle anlatılan şey Türkçede tek sözcüğe sığar:

> *house* / *ev* → *in my houses* / *evlerimde* (ev-ler-im-de = ev-ÇOĞUL-1TEKİL.İYELİK-BULUNMA)

İyi bir tokenizer biçimbirim benzeri sınırları bulabilmelidir.

In [ ]:
TR_CORPUS = ("ev evler evim evimde evlerim evlerimde " * 100 +
             "kitap kitabı kitabım kitaplar kitaplarım kitaplarımda " * 100 +
             "gel geldim geldin geliyorum geleceğim gelecektim " * 100).lower()

bpe_tr = BPETokenizer().train(TR_CORPUS, num_merges=80)
wp_tr  = WordPieceTokenizer().train(TR_CORPUS, num_merges=80)
uni_tr = UnigramTokenizer(vocab_size=60).train(TR_CORPUS, num_iters=4)

test_words = ['evlerimde', 'kitaplarım', 'geleceğim']
print(f"{'sözcük':<15} {'BPE':<35} {'WordPiece':<35} {'Unigram'}")
for w in test_words:
    print(f"{w:<15} {str(bpe_tr.encode_word(w)):<35} "
          f"{str(wp_tr.encode_word(w)):<35} "
          f"{uni_tr.encode_word(w)}")

## 6. Kıyaslama: sıfırdan yazılan ve Hugging Face `tokenizers`

`tokenizers` kuruluysa bu hücre hız ve sıkıştırmayı kıyaslar; değilse sorunsuzca atlar. Karşılaştırmanın adil olması için her iki tokenizer da önce eğitilir, ardından **yalnızca kodlama süresi** ölçülür.

In [ ]:
import time
try:
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace

    hf = Tokenizer(BPE(unk_token='[UNK]'))
    hf.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=500, special_tokens=['[UNK]'])
    hf.train_from_iterator([big_corpus], trainer)

    ours_tok = BPETokenizer().train(big_corpus, num_merges=500)  # eğitim süreye dâhil değil

    text = big_corpus[:5000]
    t0 = time.perf_counter(); ours = ours_tok.encode(text); t1 = time.perf_counter()
    t2 = time.perf_counter(); theirs = hf.encode(text).tokens; t3 = time.perf_counter()
    print(f"Bizimki: {len(ours):5d} token, {(t1-t0)*1000:6.1f} ms (yalnızca kodlama)")
    print(f"HF     : {len(theirs):5d} token, {(t3-t2)*1000:6.1f} ms (yalnızca kodlama)")
    print("Not: bizim BPE sözcük sonu işareti </w>'yi ayrı token sayabilir; token sayıları birebir karşılaştırılamaz.")
except ImportError:
    print("`tokenizers` kurulu değil — kıyaslama atlanıyor.")
    print("Kurmak için: pip install tokenizers")

## 7. Alıştırmalar

1. **Sıkıştırma eğrisi.** 10 MB'lık bir metin üzerinde $V \in \{500, 2000, 8000, 32000\}$ ile BPE eğitin. Cümle başına ortalama token sayısını $V$'ye göre çizdirin. Azalan getiri hangi $V$'de başlıyor?
2. **Bayt düzeyinde BPE.** GPT-2 karakterler yerine baytlar üzerinde çalışır — bu, hiçbir zaman `[UNK]` tokenı oluşmamasını garanti eder. Bayt düzeyinde bir varyant uygulayın ve emoji ile kod dâhil keyfî metinleri kodlayabildiğini doğrulayın.
3. **Türkçe biçimbirim kurtarma.** 20 Türkçe sözcüğün biçimbirim sınırlarını elle etiketleyin. Her tokenizer'ın bunları ne sıklıkla bulduğunu ölçün. Hangisi kazanıyor, neden?
4. **EM ile Unigram dil modeli.** Yukarıdaki uygulama Viterbi (sert EM) kullanıyor. Bunu ileri algoritmasıyla (yumuşak EM) değiştirin ve yakınsamayı karşılaştırın.

---

## Gelecek Hafta

Hafta 3 — Klasik Metin Temsilleri. Sözcük torbası, TF-IDF, n-gram dil modelleri ve LSA; hepsi NumPy ile.